In [0]:
%sql
create schema if not exists catalog_southeastasia_mdm_share_prod.share_mdm_config;

In [0]:
%sql
use catalog_southeastasia_mdm_share_prod.share_mdm_config;

In [0]:
%sql
CREATE OR REPLACE TABLE t_kafka_last_read (
  task_id STRING, 
  batch_id STRING, 
  topic STRING, 
  last_end_ms BIGINT, 
  last_end_dt TIMESTAMP,
  read_record_count LONG,
  creation_time TIMESTAMP
)

In [0]:
%sql
CREATE OR REPLACE TABLE t_topic_batch_log (
  task_id STRING, 
  batch_id STRING, 
  topic STRING, 
  batch_number STRING, 
  read_record_count LONG,
  creation_time TIMESTAMP,
  is_handle BOOLEAN COMMENT 'FALSE:未处理; TRUE:已处理'
)

In [0]:
%sql
CREATE OR REPLACE TABLE t_task_batchlist_log (
    Task_id STRING COMMENT '任务ID（外部传入）',
    Batch_id_list STRING COMMENT '分配的Batch ID列表，逗号分隔',
    Task_process_count BIGINT COMMENT '实际处理的记录总数',
    Topic_Process_Count STRING COMMENT "每个Topic实际处理的记录总数",
    Max_Task_Process_count BIGINT COMMENT '单次任务最大处理量（配额上限）',
    Create_Time TIMESTAMP COMMENT '任务创建时间戳'
)

In [0]:
%sql
CREATE OR REPLACE TABLE t_topic_priority (
    Topic_Name STRING COMMENT "Topic名称",
    Priority INT COMMENT "优先级数值,默认值1（越大越高）",
    Topic_Max_Process_Count LONG COMMENT "Topic一次任务最大处理条数",
    Is_Active INT COMMENT "是否启用（1=启用，0=未启用）"
)

In [0]:
%sql
-- Step2配置表：支持对consumer, phone, address, email表进行配置, 一个market可能有多条配置
CREATE OR REPLACE TABLE t_clean_condition (
    market_code STRING,
    type STRING COMMENT '处理类型(Consumer, Email, Phone, Address)',
    condition STRING COMMENT '需要加的过滤条件',
    is_active BOOLEAN COMMENT '该条件是否生效',
    create_time TIMESTAMP COMMENT '创建时间'
)

In [0]:
%sql
-- 重试配置表：support脚本将待重试srcc_id写入此表，重试workflow在Step2按此表筛选数据
CREATE OR REPLACE TABLE retry_config (
  task_id STRING COMMENT '重试任务ID，一次重试操作生成一批相同重试任务ID的数据',
  market_code STRING COMMENT '市场编码',
  srcc_id STRING COMMENT '待重试源数据主键',
  condition_str STRING COMMENT 'support输入的筛选条件',
  retry_type STRING COMMENT '重试类型: main-主流程任务重试 / standard-数据清洗任务重试',
  retry_status STRING COMMENT '重试状态: PENDING / SUCCESS / FAILED',
  retry_target STRING COMMENT '重试目标：发送到的Kafka Topic名称',
  create_time TIMESTAMP COMMENT '创建时间',
  create_user STRING COMMENT '创建人',
  update_time TIMESTAMP COMMENT '更新时间'
)

In [0]:
%sql
-- 下游重发配置表：support脚本全量读取本表的 MarketCode + MDMKey，从dataset回查JSON并重发Kafka
CREATE OR REPLACE TABLE retry_config_downstream (
  MarketCode STRING COMMENT '市场编码',
  MDMKey STRING COMMENT '消费者主键（MDMKey）'
)

In [0]:
%sql
-- tmatchexcludeconsumer
CREATE OR REPLACE TABLE t_merge_exclude_consumer_config (
    tmec_id STRING,
    tmec_marketcode STRING,
    tmec_type STRING, -- LineBind, ACS, Rakuten, Linegift
    tmec_sourcesystemcode STRING,
    PRIMARY KEY (tmec_marketcode, tmec_type, tmec_sourcesystemcode)
)

In [0]:
%sql
-- sconsumermediarejects
CREATE OR REPLACE TABLE t_merge_exclude_media_config (
  MarketCode STRING,
  mediaAddress STRING
)

In [0]:
%sql
-- tmatchexcludephone
CREATE OR REPLACE TABLE t_merge_exclude_phone_config (
  MarketCode STRING,
  phoneNumber STRING
)


In [0]:
%sql
-- tmatchexcludeaddress
CREATE OR REPLACE TABLE t_merge_exclude_address_config (
  MarketCode STRING,
  Address1 STRING
)

In [0]:
%sql
-- for exclusion of step5 acsoptinlist
CREATE OR REPLACE TABLE t_merge_exclude_config (
  MarketCode STRING,
  exclude_type STRING, -- mediaAddress, phoneNumber
  exclude_value STRING,
  PRIMARY KEY (MarketCode, exclude_type, exclude_value)
)

In [0]:
%sql
CREATE OR REPLACE TABLE t_survive_exclude_ukey_config (
    marketcode STRING,
    consumermdmkey STRING
)
USING DELTA;

In [0]:
%sql
-- tprogrammappingtodtl
CREATE OR REPLACE TABLE t_membership_program_code (
  MarketCode STRING,
  prgt_code STRING 
) COMMENT 'kOR SSG&kakao program_code'

In [0]:
%sql
-- tcbrexcludeprogram
CREATE OR REPLACE TABLE t_cbr_exclude_program_code (
  MarketCode STRING,
  prgt_code STRING 
) COMMENT 'prgt_code排除表 (用于生成cbr时排除program,目前仅限KOR)'

In [0]:
%sql
CREATE OR REPLACE TABLE  t_task_step_log (
  id         STRING      NOT NULL COMMENT 'uuid',
  project    STRING      NOT NULL COMMENT 'fixed: consumerlist',
  task_id    STRING      COMMENT 'pipeline task id',
  step_num   STRING      NOT NULL COMMENT '01~07',
  step_name  STRING      NOT NULL COMMENT 'python file name',
  start_time TIMESTAMP   COMMENT 'file start time',
  end_time   TIMESTAMP   COMMENT 'file end time',
  status     STRING      NOT NULL COMMENT 'SUCCESS/FAILED/WARNING',
  message    STRING      COMMENT 'summary or error'
)

In [0]:
%sql

CREATE or replace TABLE downstream_config
(
    config_id BIGINT COMMENT 'config id',
    downstream_type STRING COMMENT '写入类型: sftp',
    downstream_mode STRING COMMENT '数据下发模式: TC_INCR(增量, 默认) / FULL(全量 t 表快照)',
    table_name STRING COMMENT 'source 表名. 例: catalog_southeastasia_cdp_silver_uat.test_advancecx.consumer',
    table_business_keys ARRAY<STRING> COMMENT '用于跨多版本计算增量数据的业务主键',
    marketcode STRING COMMENT 'market code. 等于 "All" 时不筛选 market',
    brandcode STRING COMMENT 'brand code. 等于 "All" 时不筛选 brand',
    condition_str STRING COMMENT 'sql语言 筛选条件. 例: market_code = "AUG" and brand_code = "01" ',

    out_file_folder STRING COMMENT '导出目录',
    out_file_name STRING COMMENT '文件名',
    is_split_file BOOLEAN COMMENT '是否切分为多个文件',
    split_row_count BIGINT COMMENT '切分条数',

    value_map MAP<STRING, STRING> COMMENT 'sql语言 字段映射. 例: map{"active": "case when num > 1 then true else 0 end "}',

    include_fields ARRAY<STRING> COMMENT '包含字段.',
    exclude_fields ARRAY<STRING> COMMENT '排除字段. 当 include_fields 不为空时忽略 exclude_fields',

    is_backups BOOLEAN COMMENT '是否进行备份',

    is_active BOOLEAN COMMENT '是否启用配置',

    is_zip BOOLEAN COMMENT '是否通过zip方式压缩文件',
    is_encrypt BOOLEAN COMMENT '是否进行文件加密',
    encrypt_config STRING COMMENT '加密配置信息. {"encrypt_type": "AES", "_other_config": ""}',

    kafka_topic_name STRING COMMENT 'topic 名称',
    
    comment_str STRING COMMENT '注释',
    create_time TIMESTAMP,
    create_user STRING,
    update_time TIMESTAMP,
    update_user STRING

)
TBLPROPERTIES (
   'delta.columnMapping.mode' = 'name'
)

In [0]:
%sql

CREATE  TABLE downstream_log
(
  task_id STRING COMMENT 'uuid',
  config_id LONG COMMENT 'config id',

  downstream_file_num LONG COMMENT '下发文件数',
  downstream_row_count LONG COMMENT '下发条数',
  downstream_file_list ARRAY<STRING> COMMENT '下发文件列表',

  first_cdc_operation_time TIMESTAMP COMMENT '增量数据起始版本, 不包含该版本数据',
  second_cdc_operation_time TIMESTAMP COMMENT '增量数据最终版本, 包含该版本数据',

  comment_str STRING COMMENT '',
  create_time TIMESTAMP,
  update_time TIMESTAMP
)
TBLPROPERTIES (
   'delta.columnMapping.mode' = 'name'
  )